# 04 — End-to-end demo: od `git clone` do survival dataset

Demonstracja **samowystarczalności pipeline'u**. Notebook pokazuje pełną drogę od pustego repo do gotowego do uczenia maszynowego datasetu - bez żadnego ręcznego pobierania danych z portalu GDC, bez ręcznego klikania w przeglądarce.

Krok po kroku:

1. **`luad-huba download`** - pobiera 10 testowych plików z GDC (STAR-Counts, sample_sheet, clinical, metadata.cart.json)
2. **`luad-huba parse-star`** - parsuje pliki STAR-Counts do formatu parquet
3. **`luad-huba validate-cohort`** - kontrola jakości kohorty (raport JSON)
4. **`luad-huba build-matrix`** - buduje macierz ekspresji (z konfiguracją YAML)
5. **`luad-huba build-survival`** - buduje finalny survival dataset (z konfiguracją YAML)
6. Sanity check finalnego datasetu

**Notebook pobiera 10 plików (~40 MB) do katalogu tymczasowego** - nie zaśmieca `data/raw/`, bezpieczny do wielokrotnego odpalania. Dla pełnej kohorty wystarczy zwiększyć `SIZE = 601` lub usunąć `--size`.

**Cel notebooka:** dowód że pipeline jest naprawdę samowystarczalny. Jeśli wszystkie 5 kroków przejdzie bez błędu, pipeline jest gotowy do produkcyjnego użycia na pełnej kohorcie.


In [1]:
import subprocess
import sys
import tempfile
import shutil
from pathlib import Path
from datetime import datetime, timezone

import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

DEMO_DIR = Path(tempfile.mkdtemp(prefix="luad_huba_demo_"))
DATA_RAW = DEMO_DIR / "raw"
DATA_INTERIM = DEMO_DIR / "interim" / "star_counts"
DATA_PROCESSED = DEMO_DIR / "processed"
LOGS_QC = DEMO_DIR / "logs" / "qc"

SIZE = 10

print(f"Projekt: {PROJECT_ROOT}")
print(f"Katalog demo (tymczasowy): {DEMO_DIR}")
print(f"Polars: {pl.__version__}")
print(f"Czas: {datetime.now(timezone.utc).isoformat()}")
print(f"Plików do pobrania w tym demo: {SIZE}")


Projekt: /Users/luka/luad-huba-clean
Katalog demo (tymczasowy): /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03
Polars: 1.40.1
Czas: 2026-06-09T06:03:36.321928+00:00
Plików do pobrania w tym demo: 10


## Krok 1 — pobranie kohorty z GDC

`luad-huba download` jest pojedynczą komendą, która spina:
- zapytanie do `/files` o metadane plików STAR
- zapis `gdc_sample_sheet.tsv` w formacie portalu
- zapis `metadata.cart.json` z pełnymi metadanymi
- zapytanie do `/cases` o dane kliniczne
- zapis `clinical.tsv`
- pobranie samych plików STAR z weryfikacją MD5

Wynik: 4 typy plików, dokładnie te same które dostalibyśmy z UI portalu.


In [2]:
def run_cli(args, cwd=PROJECT_ROOT):
    """Uruchamia komendę luad-huba i wypisuje jej output. Rzuca CalledProcessError przy błędzie."""
    print(f"$ python -m src.cli {' '.join(str(a) for a in args)}")
    result = subprocess.run(
        [sys.executable, "-m", "src.cli", *args],
        cwd=cwd,
        capture_output=True,
        text=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Komenda zakończona z kodem {result.returncode}")
    return result


run_cli([
    "download",
    "--output-dir", str(DATA_RAW),
    "--size", str(SIZE),
])


$ python -m src.cli download --output-dir /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw --size 10
=== Pobieranie kohorty z GDC: project=TCGA-LUAD, workflow='STAR - Counts' ===
Limit liczby plików: 10

[1/4] Zapytanie o metadane plików...
  Otrzymano metadane dla 10 plików (z 601 dostępnych)

[2/4] Zapis sample_sheet.tsv i metadata.cart.json...
  Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/gdc_sample_sheet.tsv
  Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/metadata.cart.json

[3/4] Zapytanie o dane kliniczne (/cases)...
  Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/clinical.tsv (943 wierszy, 585 pacjentów)

[4/4] Pobieranie 10 plików STAR-Counts (~40 MB)...
  Pobrano: 10/10 plików zweryfikowanych

=== Kohorta gotowa w /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw ===




Pobieranie z GDC: 100%|██████████| 10/10 [00:36<00:00,  3.63s/plik]



CompletedProcess(args=['/opt/miniconda3/bin/python', '-m', 'src.cli', 'download', '--output-dir', '/var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw', '--size', '10'], returncode=0, stdout="=== Pobieranie kohorty z GDC: project=TCGA-LUAD, workflow='STAR - Counts' ===\nLimit liczby plików: 10\n\n[1/4] Zapytanie o metadane plików...\n  Otrzymano metadane dla 10 plików (z 601 dostępnych)\n\n[2/4] Zapis sample_sheet.tsv i metadata.cart.json...\n  Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/gdc_sample_sheet.tsv\n  Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/metadata.cart.json\n\n[3/4] Zapytanie o dane kliniczne (/cases)...\n  Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/clinical.tsv (943 wierszy, 585 pacjentów)\n\n[4/4] Pobieranie 10 plików STAR-Counts (~40 MB)...\n  Pobrano: 10/10 plików zweryfikowanych\n\n=== Kohorta gotowa w /var/folde

In [3]:
# Co rzeczywiście wpadło do DATA_RAW
print("=== Pliki w data/raw/ po download ===")
for path in sorted(DATA_RAW.iterdir()):
    size_mb = path.stat().st_size / 1024**2
    print(f"  {path.name:60} {size_mb:>8.2f} MB")


=== Pliki w data/raw/ po download ===
  11d52676-7017-48b4-872a-cefb91b8651c.rna_seq.augmented_star_gene_counts.tsv     4.05 MB
  31a238a8-4238-4d56-aac5-4769ce173664.rna_seq.augmented_star_gene_counts.tsv     4.02 MB
  461cd8f2-fe53-448d-b6a9-0a95792464c7.rna_seq.augmented_star_gene_counts.tsv     4.04 MB
  6200626d-556e-400d-81f4-e397ce49585f.rna_seq.augmented_star_gene_counts.tsv     4.04 MB
  743f08af-5537-4454-a219-18639e5127c6.rna_seq.augmented_star_gene_counts.tsv     4.05 MB
  afccc865-1d58-475d-b85c-0d7eca330b0e.rna_seq.augmented_star_gene_counts.tsv     4.02 MB
  b7f29b8c-08a8-4781-ba33-5a7b6b02ad23.rna_seq.augmented_star_gene_counts.tsv     4.04 MB
  c812eb06-bd41-4919-8f6d-9ade00654a4c.rna_seq.augmented_star_gene_counts.tsv     4.04 MB
  clinical.tsv                                                     0.06 MB
  ebf8d429-0437-4987-9bca-89e62910f168.rna_seq.augmented_star_gene_counts.tsv     4.05 MB
  ef11ddff-02bb-44e2-ab77-fadde20acd87.rna_seq.augmented_star_gene_counts.tsv

## Krok 2 — parsowanie STAR-Counts

Każdy plik TSV jest przekształcany do parquet (kolumnowy format, szybsze wczytywanie, mniejsze pliki).


In [4]:
run_cli([
    "parse-star",
    "--input-dir", str(DATA_RAW),
    "--output-dir", str(DATA_INTERIM),
])

print()
print(f"=== Liczba parquetów: {len(list(DATA_INTERIM.glob('*.parquet')))} ===")


$ python -m src.cli parse-star --input-dir /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw --output-dir /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/interim/star_counts
Znaleziono 10 plik(i) STAR-Counts do przetworzenia.
[1/10] Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/interim/star_counts/11d52676-7017-48b4-872a-cefb91b8651c.parquet
[2/10] Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/interim/star_counts/31a238a8-4238-4d56-aac5-4769ce173664.parquet
[3/10] Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/interim/star_counts/461cd8f2-fe53-448d-b6a9-0a95792464c7.parquet
[4/10] Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/interim/star_counts/6200626d-556e-400d-81f4-e397ce49585f.parquet
[5/10] Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/interim/star

## Krok 3 — walidacja kohorty

Cztery reguły QC sprawdzające spójność: brak plików, orphan files, brak clinical, duplikaty próbek. Wynik to JSON ze stemplem czasowym UTC.


In [5]:
run_cli([
    "validate-cohort",
    "--sample-sheet", str(DATA_RAW / "gdc_sample_sheet.tsv"),
    "--clinical", str(DATA_RAW / "clinical.tsv"),
    "--interim-dir", str(DATA_INTERIM),
    "--log-dir", str(LOGS_QC),
])

reports = sorted(LOGS_QC.glob("qc_report_*.json"))
print()
print(f"Raport QC zapisany: {reports[-1] if reports else 'BRAK'}")


$ python -m src.cli validate-cohort --sample-sheet /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/gdc_sample_sheet.tsv --clinical /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/clinical.tsv --interim-dir /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/interim/star_counts --log-dir /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/logs/qc
Wczytuję sample sheet: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/gdc_sample_sheet.tsv
Wczytuję dane kliniczne: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/clinical.tsv
Skanuję katalog plików: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/interim/star_counts
Znaleziono 10 plik(ów) w /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/interim/star_counts

=== Podsumowanie QC ===
  Wszystkie problemy: 0
  ERROR:    0
  INFO:     0

Rap

clinical_parser: pominięto 9 pacjentów bez czasu obserwacji w clinical.tsv (przykłady: ['TCGA-75-6207', 'TCGA-75-5122', 'TCGA-75-6205']). Sprawdź days_to_death oraz days_to_last_follow_up.



## Krok 4 — budowa macierzy ekspresji

Tu używamy **flagi `--config`** żeby pokazać że konfiguracja YAML faktycznie działa. `configs/default.yaml` mówi `normalization.method: tpm` - macierz zostanie zbudowana z metryki TPM zamiast raw counts.


In [6]:
run_cli([
    "build-matrix",
    "--input-dir", str(DATA_INTERIM),
    "--sample-sheet", str(DATA_RAW / "gdc_sample_sheet.tsv"),
    "--output-dir", str(DATA_PROCESSED),
    "--config", str(PROJECT_ROOT / "configs" / "default.yaml"),
    "--duplicate-strategy", "deepest",
])

matrix_path = DATA_PROCESSED / "expression_matrix.parquet"
print()
print(f"=== Macierz zapisana: {matrix_path} ===")
print(f"Rozmiar: {matrix_path.stat().st_size / 1024**2:.2f} MB")


$ python -m src.cli build-matrix --input-dir /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/interim/star_counts --sample-sheet /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/gdc_sample_sheet.tsv --output-dir /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/processed --config /Users/luka/luad-huba-clean/configs/default.yaml --duplicate-strategy deepest
Załadowano config: /Users/luka/luad-huba-clean/configs/default.yaml
Metryka z configu: 'tpm' -> 'tpm_unstranded'
Wczytuję sample sheet: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/gdc_sample_sheet.tsv
Buduję macierz z 10 plik(ów) parquet, metryka: tpm_unstranded
Macierz zapisana: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/processed/expression_matrix.parquet (60660 genów x 10 próbek)
Manifest zapisany: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/processed/expression_matr

## Krok 5 — budowa survival dataset

Druga komenda używająca konfiguracji YAML. Tu działa `survival.min_follow_up_days: 30` - próbki o czasie obserwacji krótszym niż 30 dni zostaną odfiltrowane.


In [7]:
run_cli([
    "build-survival",
    "--matrix", str(DATA_PROCESSED / "expression_matrix.parquet"),
    "--sample-sheet", str(DATA_RAW / "gdc_sample_sheet.tsv"),
    "--clinical", str(DATA_RAW / "clinical.tsv"),
    "--output-dir", str(DATA_PROCESSED),
    "--config", str(PROJECT_ROOT / "configs" / "default.yaml"),
])

survival_path = DATA_PROCESSED / "survival_dataset.parquet"
print()
print(f"=== Survival dataset zapisany: {survival_path} ===")
print(f"Rozmiar: {survival_path.stat().st_size / 1024**2:.2f} MB")


$ python -m src.cli build-survival --matrix /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/processed/expression_matrix.parquet --sample-sheet /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/gdc_sample_sheet.tsv --clinical /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/clinical.tsv --output-dir /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/processed --config /Users/luka/luad-huba-clean/configs/default.yaml
Załadowano config: /Users/luka/luad-huba-clean/configs/default.yaml
Wczytuję macierz: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/processed/expression_matrix.parquet
Wczytuję sample sheet: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/gdc_sample_sheet.tsv
Wczytuję dane kliniczne: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03/raw/clinical.tsv
Buduję zbiór przeżywalności (próbki: tylko nowot

clinical_parser: pominięto 9 pacjentów bez czasu obserwacji w clinical.tsv (przykłady: ['TCGA-75-5122', 'TCGA-75-6203', 'TCGA-75-5126']). Sprawdź days_to_death oraz days_to_last_follow_up.



## Krok 6 — sanity check finalnego datasetu

Czy wynik ma sens? Wczytujemy parquet, sprawdzamy rozmiar, kolumny meta, rozkład event/censoring.


In [8]:
dataset = pl.read_parquet(survival_path)
print(f"=== Dataset: {dataset.height} próbek x {dataset.width} kolumn ===")

n_metadata = len([c for c in dataset.columns if not c.startswith("ENSG")])
n_genes = len([c for c in dataset.columns if c.startswith("ENSG")])
print(f"  Kolumny metadanych: {n_metadata}")
print(f"  Kolumny genów: {n_genes}")
print()

print("Pierwsze 5 kolumn metadanych:")
print(dataset.select(dataset.columns[:n_metadata]).head(3))
print()

print("Rozkład event/censoring:")
print(dataset.group_by("event").len())


=== Dataset: 8 próbek x 60668 kolumn ===
  Kolumny metadanych: 8
  Kolumny genów: 60660

Pierwsze 5 kolumn metadanych:
shape: (3, 8)
┌───────────────┬──────────────┬──────┬───────┬──────────────┬────────┬──────────────┬─────────────┐
│ sample_id     ┆ case_id      ┆ time ┆ event ┆ age_at_index ┆ gender ┆ ajcc_patholo ┆ tissue_type │
│ ---           ┆ ---          ┆ ---  ┆ ---   ┆ ---          ┆ ---    ┆ gic_stage    ┆ ---         │
│ str           ┆ str          ┆ i64  ┆ bool  ┆ i64          ┆ str    ┆ ---          ┆ str         │
│               ┆              ┆      ┆       ┆              ┆        ┆ str          ┆             │
╞═══════════════╪══════════════╪══════╪═══════╪══════════════╪════════╪══════════════╪═════════════╡
│ TCGA-55-7227- ┆ TCGA-55-7227 ┆ 952  ┆ true  ┆ 77           ┆ male   ┆ Stage IIIA   ┆ Tumor       │
│ 01A           ┆              ┆      ┆       ┆              ┆        ┆              ┆             │
│ TCGA-73-4666- ┆ TCGA-73-4666 ┆ 800  ┆ false ┆ 52         

In [9]:
print("=== Demo zakończone pomyślnie ===")
print()
print(f"Wszystkie artefakty pipeline'u znajdują się w: {DEMO_DIR}")
print()
print("Co właśnie zostało udowodnione:")
print("1. luad-huba download pobiera kohorte z GDC bez żadnej autoryzacji")
print("2. luad-huba parse-star transformuje TSV -> parquet (szybsze IO)")
print("3. luad-huba validate-cohort produkuje raport QC w JSON")
print("4. luad-huba build-matrix --config buduje macierz wg konfiguracji YAML")
print("5. luad-huba build-survival --config produkuje ML-ready dataset")
print()
print("Pipeline jest naprawdę samowystarczalny - bez ręcznych kroków,")
print("bez ściągania niczego z przeglądarki, bez kopiowania plików.")
print()
print(f"Katalog tymczasowy {DEMO_DIR} można usunąć w dowolnym momencie:")
print(f"  shutil.rmtree({DEMO_DIR!r})")


=== Demo zakończone pomyślnie ===

Wszystkie artefakty pipeline'u znajdują się w: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03

Co właśnie zostało udowodnione:
1. luad-huba download pobiera kohorte z GDC bez żadnej autoryzacji
2. luad-huba parse-star transformuje TSV -> parquet (szybsze IO)
3. luad-huba validate-cohort produkuje raport QC w JSON
4. luad-huba build-matrix --config buduje macierz wg konfiguracji YAML
5. luad-huba build-survival --config produkuje ML-ready dataset

Pipeline jest naprawdę samowystarczalny - bez ręcznych kroków,
bez ściągania niczego z przeglądarki, bez kopiowania plików.

Katalog tymczasowy /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03 można usunąć w dowolnym momencie:
  shutil.rmtree(PosixPath('/var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_demo_ynbejg03'))


## Wnioski

Pełen pipeline LUAD-HUBA, od pustego repo do gotowego survival dataset, wymaga:

```bash
git clone https://github.com/WorthySubset151/luad-huba-clean.git
cd luad-huba-clean
uv sync
luad-huba download
luad-huba parse-star
luad-huba validate-cohort
luad-huba build-matrix --config configs/default.yaml
luad-huba build-survival --config configs/default.yaml
```

Siedem komend, zero kliknięć w przeglądarce, zero ręcznego kopiowania plików. **Repo jest samowystarczalne**.

Co dalej (poza zakresem tego notebooka):

- Notebook `05_baseline_survival` - pierwsze modele Kaplan-Meier + Cox na survival_dataset.parquet
- Strategia feature selection (60660 genów × 533 próbki → top variable / LASSO / panel ekspercki)
- Warstwa modelowania (`src/models/`)
- Interfejs Streamlit (`src/app/`) - po zbudowaniu modeli
